# ShopSense: when will each customer pay next, and how much?

Data: dad's credit book in MySQL (`shopsense.ledger`, `shopsense.customers`).

Plan:
1. Load the data from MySQL into pandas
2. Look at each customer's payment history (gaps between payments, usual amount)
3. Predict next payment date and amount with a simple rule
4. Test the rule on payments we already know (hide the last one, predict it)

In [9]:
import pandas as pd
import pymysql
from getpass import getpass

conn = pymysql.connect(host="localhost", user="root",
                       password=getpass("MySQL password: "),
                       database="shopsense")

df = pd.read_sql("SELECT * FROM ledger", conn)
print(df.shape)
df.head()

(3065, 8)


C:\Users\chethan\AppData\Local\Temp\ipykernel_40296\312848640.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM ledger", conn)


,entry_id,entry_date,customer_id,entry_type,amount,written_balance,page_no,notes
0,1,2024-11-30,CUST_001,Purchase,202.0,NaN,82,Opening balance 0: earlier BAKI 985 was paid b...
1,2,2024-11-30,CUST_001,Purchase,100.0,NaN,82,None
2,3,2024-12-02,CUST_001,Purchase,70.0,NaN,82,None
3,4,2024-12-05,CUST_001,Purchase,2348.0,NaN,82,Checked in book
4,5,2024-12-08,CUST_001,Purchase,79.0,NaN,82,None


In [10]:
pay = df[df["entry_type"] == "Payment"].dropna(subset=["entry_date"]).copy()
pay["entry_date"] = pd.to_datetime(pay["entry_date"])
print(len(pay))

summary = pay.groupby("customer_id")["amount"].agg(["count", "mean"])
summary.sort_values("count", ascending=False).head()


98


,count,mean
customer_id,,
CUST_021,21,833.333333
CUST_027,6,9733.333333
CUST_026,6,3654.500000
CUST_011,5,1137.800000
CUST_007,5,8679.200000


In [11]:
pay = pay.sort_values(["customer_id", "entry_date"])
pay["gap_days"] = pay.groupby("customer_id")["entry_date"].diff().dt.days

pay[pay["customer_id"] == "CUST_021"][["entry_date", "amount", "gap_days"]].head(6)


,entry_date,amount,gap_days
2326,2022-09-07,3000.0,NaN
2340,2022-10-01,1000.0,24.0
2342,2022-10-03,1000.0,2.0
2343,2022-10-06,1000.0,3.0
2366,2022-11-05,800.0,30.0
2368,2022-11-07,400.0,2.0


In [12]:
rule = pay.groupby("customer_id").agg(
    payments=("amount", "count"),
    last_payment=("entry_date", "max"),
    avg_gap=("gap_days", "mean"),
    usual_amount=("amount", "median"),
)
rule["next_payment_date"] = rule["last_payment"] + pd.to_timedelta(rule["avg_gap"].round(), unit="D")

rule.sort_values("payments", ascending=False).head(8)


,payments,last_payment,avg_gap,usual_amount,next_payment_date
customer_id,,,,,
CUST_021,21,2023-03-02,8.80,600.0,2023-03-11
CUST_027,6,2024-07-12,127.80,8300.0,2024-11-17
CUST_026,6,2023-03-07,36.00,3713.5,2023-04-12
CUST_011,5,2023-03-11,29.00,1000.0,2023-04-09
CUST_007,5,2022-12-12,22.75,9788.0,2023-01-04
CUST_006,4,2023-02-18,52.00,5711.0,2023-04-11
CUST_029,4,2025-02-10,12.00,3500.0,2025-02-22
CUST_017,4,2023-01-10,30.00,6500.0,2023-02-09


In [13]:
rows = []
for cust, g in pay.groupby("customer_id"):
    if len(g) < 3:
        continue                                  # need at least 2 gaps to learn from
    history = g.iloc[:-1]                         # all payments except the last
    actual = g.iloc[-1]                           # the last payment (hidden)
    gap = history["entry_date"].diff().dt.days.mean()
    predicted_date = history["entry_date"].max() + pd.Timedelta(days=round(gap))
    rows.append({
        "customer_id": cust,
        "predicted_date": predicted_date,
        "actual_date": actual["entry_date"],
        "days_off": abs((actual["entry_date"] - predicted_date).days),
        "predicted_amount": history["amount"].median(),
        "actual_amount": actual["amount"],
    })

test = pd.DataFrame(rows)
print("customers tested:", len(test))
print("average days off:", round(test["days_off"].mean(), 1))
print("median days off:", test["days_off"].median())
test

customers tested: 13
average days off: 27.4
median days off: 18.0


,customer_id,predicted_date,actual_date,days_off,predicted_amount,actual_amount
0,CUST_003,2022-12-04,2022-12-11,7,6500.0,3000.0
1,CUST_004,2023-01-13,2023-01-13,0,2000.0,5000.0
2,CUST_005,2023-03-16,2023-03-14,2,650.0,1200.0
3,CUST_006,2022-12-11,2023-02-18,69,5336.0,6832.0
4,CUST_007,2022-12-01,2022-12-12,11,9894.0,9013.0
5,CUST_010,2023-08-26,2023-06-01,86,1800.0,1500.0
6,CUST_011,2023-01-31,2023-03-11,39,907.0,1166.0
7,CUST_015,2024-05-21,2024-05-03,18,4800.0,4856.0
8,CUST_017,2022-11-26,2023-01-10,45,6000.0,7000.0
9,CUST_021,2023-02-20,2023-03-02,10,550.0,1000.0


In [14]:
prev = pay.groupby("customer_id")["entry_date"].agg(lambda s: s.iloc[-2] if len(s) > 1 else pd.NaT)

test["naive_date"] = test["customer_id"].map(prev) + pd.Timedelta(days=30)
test["naive_days_off"] = (test["actual_date"] - test["naive_date"]).dt.days.abs()

print("Our rule  - average:", round(test["days_off"].mean(), 1), " median:", test["days_off"].median())
print("30-day guess - average:", round(test["naive_days_off"].mean(), 1), " median:", test["naive_days_off"].median())

Our rule  - average: 27.4  median: 18.0
30-day guess - average: 24.4  median: 13.0


In [15]:
rule["confidence"] = pd.cut(rule["payments"], bins=[0, 2, 4, 100],
                            labels=["Low", "Medium", "High"])

final = rule[["payments", "last_payment", "next_payment_date", "usual_amount", "confidence"]]
final.to_csv("payment_predictions.csv")
final.sort_values("payments", ascending=False)

,payments,last_payment,next_payment_date,usual_amount,confidence
customer_id,,,,,
CUST_021,21,2023-03-02,2023-03-11,600.0,High
CUST_027,6,2024-07-12,2024-11-17,8300.0,High
CUST_026,6,2023-03-07,2023-04-12,3713.5,High
CUST_011,5,2023-03-11,2023-04-09,1000.0,High
CUST_007,5,2022-12-12,2023-01-04,9788.0,High
CUST_006,4,2023-02-18,2023-04-11,5711.0,Medium
CUST_029,4,2025-02-10,2025-02-22,3500.0,Medium
CUST_017,4,2023-01-10,2023-02-09,6500.0,Medium
CUST_015,4,2024-05-03,2024-06-12,4828.0,Medium
